# 05 — Cross-fitted weighted ensemble

Notebook 04で保存した5モデルのOOF/test予測を統合します。
重みを学習した同じOOFで評価するとスコアが楽観的になるため、ここでは各foldをhold-outし、
残り4 foldだけで重みを最適化するcross-fittingを行います。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from train import MODEL_ORDER

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN
ARTIFACT_DIR = ROOT / "artifacts"
PREFIXES = {name: name.lower().replace(" ", "_") for name in MODEL_ORDER}

## 予測を読み込み、行対応を検証

ID・目的変数・fold・test順のどれかが違えばアンサンブルを中断します。

In [ ]:
oof_frames = {}
test_frames = {}
for model_name in MODEL_ORDER:
    prefix = PREFIXES[model_name]
    oof_frames[model_name] = pd.read_csv(ARTIFACT_DIR / f"{prefix}_oof.csv")
    test_frames[model_name] = pd.read_csv(ARTIFACT_DIR / f"{prefix}_test.csv")

reference_oof = oof_frames[MODEL_ORDER[0]]
reference_test = test_frames[MODEL_ORDER[0]]
for model_name in MODEL_ORDER[1:]:
    for column in [ID_COLUMN, TARGET, FOLD_COLUMN]:
        assert np.array_equal(
            reference_oof[column].to_numpy(),
            oof_frames[model_name][column].to_numpy(),
        ), f"OOF {column} mismatch: {model_name}"
    assert np.array_equal(
        reference_test[ID_COLUMN].to_numpy(),
        test_frames[model_name][ID_COLUMN].to_numpy(),
    ), f"test ID mismatch: {model_name}"
print("OOF/test alignment: OK")

In [ ]:
y_true = reference_oof[TARGET].to_numpy()
fold_ids = reference_oof[FOLD_COLUMN].to_numpy()
oof_matrix = np.column_stack(
    [oof_frames[name]["oof_prob"].to_numpy() for name in MODEL_ORDER]
)
test_matrix = np.column_stack(
    [test_frames[name]["pred_prob"].to_numpy() for name in MODEL_ORDER]
)

score_table = pd.DataFrame(
    {
        "model": MODEL_ORDER,
        "oof_auc": [
            roc_auc_score(y_true, oof_matrix[:, index])
            for index in range(len(MODEL_ORDER))
        ],
    }
).sort_values("oof_auc", ascending=False)
display(score_table)
display(pd.DataFrame(oof_matrix, columns=MODEL_ORDER).corr())

## fold外で重みを学習

非負かつ合計1の制約を付け、AUCを最大化します。各validation foldの重みは、そのfoldの目的変数を見ずに決まります。

In [ ]:
def normalize_weights(raw_weights):
    weights = np.asarray(raw_weights, dtype=float)
    total = weights.sum()
    return weights / total if total > 0 else np.ones_like(weights) / len(weights)


def optimize_weights(predictions, target, seed):
    def objective(raw_weights):
        weights = normalize_weights(raw_weights)
        return -roc_auc_score(target, predictions @ weights)

    result = differential_evolution(
        objective,
        bounds=[(0.0, 1.0)] * predictions.shape[1],
        seed=seed,
        maxiter=50,
        popsize=10,
        tol=1e-7,
        polish=True,
        workers=1,
        updating="immediate",
    )
    return normalize_weights(result.x)


cross_fitted_oof = np.zeros(len(y_true), dtype=float)
fold_weights = []
for fold in sorted(np.unique(fold_ids)):
    fit_mask = fold_ids != fold
    valid_mask = fold_ids == fold
    weights = optimize_weights(
        oof_matrix[fit_mask], y_true[fit_mask], Baseline.SEED + int(fold)
    )
    cross_fitted_oof[valid_mask] = oof_matrix[valid_mask] @ weights
    fold_weights.append(weights)

fold_weights = np.vstack(fold_weights)
test_weights = fold_weights.mean(axis=0)
weighted_test_pred = test_matrix @ test_weights

equal_oof = oof_matrix.mean(axis=1)
equal_auc = roc_auc_score(y_true, equal_oof)
cross_fitted_auc = roc_auc_score(y_true, cross_fitted_oof)
print(f"equal-weight OOF AUC       : {equal_auc:.6f}")
print(f"cross-fitted weighted AUC : {cross_fitted_auc:.6f}")

## 重みの安定性

foldごとの重みが大きく変わる場合、最適化が不安定です。その場合は単純平均を優先します。

In [ ]:
weights_by_fold = pd.DataFrame(fold_weights, columns=MODEL_ORDER)
weights_by_fold.index.name = "held_out_fold"
display(weights_by_fold)

weights_summary = pd.DataFrame(
    {
        "model": MODEL_ORDER,
        "mean_weight": test_weights,
        "std_weight": fold_weights.std(axis=0),
    }
).sort_values("mean_weight", ascending=False)
display(weights_summary)
weights_summary.plot.bar(x="model", y="mean_weight", yerr="std_weight", legend=False)
plt.ylabel("weight")
plt.title("Cross-fitted ensemble weights")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## submissionと監査用ファイル

weighted blendが単純平均を上回らない場合は、より単純な単純平均を採用します。

In [ ]:
use_weighted = cross_fitted_auc > equal_auc
final_test_pred = weighted_test_pred if use_weighted else test_matrix.mean(axis=1)
method = "cross_fitted_weighted" if use_weighted else "equal_weight"

submission = pd.DataFrame(
    {ID_COLUMN: reference_test[ID_COLUMN].to_numpy(), TARGET: final_test_pred}
)
assert submission[TARGET].between(0, 1).all()
submission_path = ROOT / "submission_weighted_ensemble.csv"
submission.to_csv(submission_path, index=False)

weights_by_fold.to_csv(ARTIFACT_DIR / "ensemble_weights_by_fold.csv")
weights_summary.to_csv(ARTIFACT_DIR / "ensemble_weights_summary.csv", index=False)
pd.DataFrame(
    [
        {
            "selected_method": method,
            "equal_weight_oof_auc": equal_auc,
            "cross_fitted_weighted_oof_auc": cross_fitted_auc,
        }
    ]
).to_csv(ARTIFACT_DIR / "ensemble_comparison.csv", index=False)
print("selected:", method)
print("saved:", submission_path)
display(submission.head())